# 03. PromptTemplate과 Message

PromptTemplate을 사용하면 프롬프트를 재사용 가능한 템플릿으로 관리할 수 있습니다.

In [1]:
import sys
sys.path.insert(0, "..")

from common_config import llm_connect

llm = llm_connect(
    model="gpt-5.4-mini",
    temperature=0,
    max_tokens=2086,
)

In [7]:
# from dotenv import load_dotenv
# load_dotenv(override=True, dotenv_path="../.env")

## 1. PromptTemplate - 단일 문자열 템플릿

In [8]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("{topic}에 대해 한 문장으로 설명해줘")

# 변수 채우기
prompt = template.invoke({"topic": "RAG"})
print(prompt)

text='RAG에 대해 한 문장으로 설명해줘'


## 2. ChatPromptTemplate - 역할 기반 템플릿

시스템/사용자/AI 메시지를 조합하여 템플릿을 만듭니다.

In [9]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}")
])

messages = chat_prompt.invoke({
    "role": "파이썬",
    "question": "데코레이터가 뭔가요?"
})

print(messages)

messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='데코레이터가 뭔가요?', additional_kwargs={}, response_metadata={})]


## 3. LLM과 연결하여 호출

In [11]:
from langchain_openai import ChatOpenAI

messages = chat_prompt.invoke({
    "role": "머신러닝",
    "question": "과적합이 뭔가요?"
})

response = llm.invoke(messages)
print(response.content)

과적합(overfitting)은 **모델이 학습 데이터에 너무 맞춰져서, 새 데이터에 대한 일반화 성능이 떨어지는 현상**입니다.

쉽게 말해:
- **훈련 데이터는 거의 완벽하게 맞춤**
- 그런데 **처음 보는 데이터에서는 성능이 나쁨**

예를 들어,
- 학생이 문제집의 답만 외워서 시험을 보면
- 문제집과 똑같은 문제는 잘 풀지만
- 응용문제는 못 푸는 것과 비슷합니다.

### 왜 생기나요?
- 모델이 너무 복잡할 때
- 데이터가 너무 적을 때
- 노이즈나 우연한 패턴까지 학습할 때

### 어떻게 알 수 있나요?
- 훈련 정확도는 높은데
- 검증/테스트 정확도는 낮으면
과적합을 의심할 수 있습니다.

### 줄이는 방법
- 더 많은 데이터 사용
- 모델 단순화
- 정규화(Regularization)
- 드롭아웃(Dropout)
- 조기 종료(Early stopping)
- 데이터 증강

원하시면 제가 **과적합, 과소적합, 일반화**를 함께 비교해서 그림처럼 쉽게 설명해드릴게요.


## 4. 대화 히스토리 포함 - MessagesPlaceholder

이전 대화 내용을 프롬프트에 삽입할 때 사용합니다.

In [12]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 AI 어시스턴트입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# 이전 대화 기록
history = [
    HumanMessage(content="내 이름은 홍길동이야"),
    AIMessage(content="안녕하세요, 홍길동님!")
]

messages = chat_prompt.invoke({
    "history": history,
    "question": "내 이름이 뭐라고?"
})

response = llm.invoke(messages)
print(response.content)

홍길동님입니다.


In [13]:
# 현재 대화를 history에 추가
history.append(HumanMessage(content="내 이름이 뭐라고?"))
history.append(AIMessage(content=response.content))
history

[HumanMessage(content='내 이름은 홍길동이야', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, 홍길동님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름이 뭐라고?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='홍길동님입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## 5. 다중 변수 템플릿 예제

In [14]:
template = ChatPromptTemplate.from_messages([
    ("system", "당신은 {language} 번역가입니다."),
    ("human", "다음 문장을 번역해줘: {sentence}")
])

messages = template.invoke({
    "language": "영어",
    "sentence": "오늘 날씨가 정말 좋네요."
})

response = llm.invoke(messages)
print(response.content)

The weather is really nice today.


## 문제 해결

In [15]:
template = ChatPromptTemplate.from_messages([
    ("system", """당신은 {sorted_language}에서 {target_language}로 번역하는 번역가입니다.
      출력을 할 때 {sorted_language}로 번역된 문장과 {target_language}로 번역된 문장을 모두 보여주세요.
     
      예시:
      입력(한국어): "오늘 날씨가 정말 좋네요.
      출력(영어): The weather is really nice today. 
     """),
    ("human", "다음 문장을 번역해줘: {text}")
])

messages = template.invoke({
    "sorted_language": "한국어",
    "target_language": "영어",
    "text": "벌써 오후 3시다."
})

response = llm.invoke(messages)
print(response.content)

번역(한국어): 벌써 오후 3시다.  
Translation(English): It’s already 3 p.m.
